# Visual comparison of orientation compensation in SAR and geographic coordinates

- This is only to check the output for geocoded images.
- Numerical comparison is done in the real-data parity notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import dask
import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

from polsarpro.io import open_netcdf_beam
from polsarpro.polarisation import orientation_compensation
from polsarpro.util import S_to_T3, multilook, pauli_rgb

input_alos_slc = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
input_alos_geo = Path(
    "/data/psp/test_files/SAN_FRANCISCO_ALOS1_geocoded_T3_7_look_az.nc"
)

In [ ]:
slc = open_netcdf_beam(input_alos_slc)
geo = open_netcdf_beam(input_alos_geo)

In [ ]:
with ProgressBar():
    T3_slc = S_to_T3(slc)
    T3_slc = multilook(T3_slc, dim_az=4, dim_rg=1)
    corrected_slc, orientation_slc = orientation_compensation(T3_slc)
    corrected_slc, orientation_slc = dask.persist(
        corrected_slc, orientation_slc
    )
with ProgressBar():
    corrected_geo, orientation_geo = orientation_compensation(geo)
    corrected_geo, orientation_geo = dask.persist(
        corrected_geo, orientation_geo
    )

In [ ]:
with ProgressBar():
    rgb_slc, rgb_geo = dask.compute(
        pauli_rgb(corrected_slc), pauli_rgb(corrected_geo)
    )

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes[0, 0].imshow(
    rgb_slc.transpose("y", "x", "band")[::2], interpolation="none"
)
axes[0, 0].set_title("Corrected Pauli RGB — SAR coordinates")
axes[0, 1].imshow(
    rgb_geo.transpose("lat", "lon", "band"), interpolation="none"
)
axes[0, 1].set_title("Corrected Pauli RGB — geographic coordinates")
angle_slc = axes[1, 0].imshow(
    orientation_slc.orientation_angle[::2],
    cmap="twilight",
    vmin=-22.5,
    vmax=22.5,
    interpolation="none",
)
axes[1, 0].set_title("Orientation angle — SAR coordinates")
angle_geo = axes[1, 1].imshow(
    orientation_geo.orientation_angle,
    cmap="twilight",
    vmin=-22.5,
    vmax=22.5,
    interpolation="none",
)
axes[1, 1].set_title("Orientation angle — geographic coordinates")
for axis in axes.flat:
    axis.axis("off")
fig.colorbar(angle_slc, ax=axes[1, 0], fraction=0.046, pad=0.04)
fig.colorbar(angle_geo, ax=axes[1, 1], fraction=0.046, pad=0.04)
plt.tight_layout()